In [4]:
# 1. imports and paths

import os
import gzip
import tarfile
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

base        = 'D:/TNBC_SV_DNA_Repair'
data_dir    = os.path.join(base, 'dataset')
tables_dir  = os.path.join(base, 'results', 'tables')
figures_dir = os.path.join(base, 'results', 'figures')
scores_dir  = os.path.join(base, 'results', 'scores')

print('paths set')

paths set


In [5]:
# 2. load gips and cohort

gips_df   = pd.read_csv(os.path.join(scores_dir, 'gips_scores.csv'))
surv_tnbc = pd.read_csv(os.path.join(tables_dir, 'surv_tnbc.csv'))
sv_ev     = pd.read_csv(os.path.join(tables_dir, 'sv_evidence_per_gene.csv'), index_col=0)

tnbc_samples  = gips_df['sample'].tolist()
sv_genes      = sv_ev[sv_ev['either_sv'] == 'yes'].index.tolist()

hr_genes      = ['BRCA1','BRCA2','PALB2','RAD51','RAD51B','RAD51C','RAD51D','BRIP1','ATM','CHEK2']
cohesin_genes = ['STAG2','STAG3','SMC1A','SMC1B','RAD21','REC8']
meiosis_genes = ['HORMAD1','HORMAD2','SYCP2','SYCP3','MLH3','MSH4','MSH5']
all_panel     = hr_genes + cohesin_genes + meiosis_genes

print('gips samples:', len(tnbc_samples))
print('sv candidate genes:', len(sv_genes))
print('gips groups:', gips_df['GIPS_group'].value_counts().to_dict())

gips samples: 121
sv candidate genes: 23
gips groups: {'Low': 52, 'High': 36, 'Moderate': 33}


In [6]:
# 3. reload TCGA full expression

tcga_expr_file = os.path.join(data_dir, 'TCGA.BRCA.sampleMap_HiSeqV2_exon.gz')

# read only header to get sample names
with gzip.open(tcga_expr_file, 'rt') as f:
    header = f.readline().rstrip().split('\t')

all_cols     = header
sample_cols  = header[1:]
tnbc_cols    = [s for s in tnbc_samples if s in sample_cols]
col_indices  = [0] + [sample_cols.index(s) + 1 for s in tnbc_cols]

print('total columns in file:', len(sample_cols))
print('tnbc columns found:', len(tnbc_cols))

total columns in file: 1218
tnbc columns found: 121


In [7]:
# 4. define immune gene sets

# from Paper 1 and literature
exhaustion_genes = [
    'PDCD1','LAG3','HAVCR2','TIGIT','CTLA4',
    'TOX','NR4A1','ENTPD1','BATF','PRDM1'
]

cytotoxic_genes = [
    'GZMB','PRF1','IFNG','CD8A','CD8B',
    'GNLY','NKG7','GZMA','GZMH','KLRG1'
]

immune_suppress_genes = [
    'FOXP3','IL10','TGFB1','CD274','PDCD1LG2',
    'IDO1','ARG1','VEGFA','IL6','STAT3'
]

pdcd1_cd2_genes = ['PDCD1','CD2']

all_immune_genes = list(set(
    exhaustion_genes + cytotoxic_genes +
    immune_suppress_genes + pdcd1_cd2_genes
))

print('exhaustion genes:', len(exhaustion_genes))
print('cytotoxic genes:', len(cytotoxic_genes))
print('suppression genes:', len(immune_suppress_genes))
print('total unique immune genes:', len(all_immune_genes))

exhaustion genes: 10
cytotoxic genes: 10
suppression genes: 10
total unique immune genes: 31


In [8]:
# 5. inspect GTEx tar contents

gtex_file = os.path.join(data_dir, 'GTEx_Analysis_v10_eQTL.tar')

with tarfile.open(gtex_file, 'r') as tar:
    members = tar.getnames()

print('total files in GTEx tar:', len(members))
print('first 20 files:')
for m in members[:20]:
    print(' ', m)

total files in GTEx tar: 101
first 20 files:
  GTEx_Analysis_v10_eQTL_updated
  GTEx_Analysis_v10_eQTL_updated/Prostate.v10.eGenes.txt.gz
  GTEx_Analysis_v10_eQTL_updated/Stomach.v10.eGenes.txt.gz
  GTEx_Analysis_v10_eQTL_updated/Skin_Not_Sun_Exposed_Suprapubic.v10.eQTLs.signif_pairs.parquet
  GTEx_Analysis_v10_eQTL_updated/Spleen.v10.eGenes.txt.gz
  GTEx_Analysis_v10_eQTL_updated/Minor_Salivary_Gland.v10.eGenes.txt.gz
  GTEx_Analysis_v10_eQTL_updated/Heart_Left_Ventricle.v10.eGenes.txt.gz
  GTEx_Analysis_v10_eQTL_updated/Pituitary.v10.eQTLs.signif_pairs.parquet
  GTEx_Analysis_v10_eQTL_updated/Minor_Salivary_Gland.v10.eQTLs.signif_pairs.parquet
  GTEx_Analysis_v10_eQTL_updated/Esophagus_Mucosa.v10.eGenes.txt.gz
  GTEx_Analysis_v10_eQTL_updated/Brain_Cerebellar_Hemisphere.v10.eQTLs.signif_pairs.parquet
  GTEx_Analysis_v10_eQTL_updated/Adipose_Subcutaneous.v10.eQTLs.signif_pairs.parquet
  GTEx_Analysis_v10_eQTL_updated/Adrenal_Gland.v10.eGenes.txt.gz
  GTEx_Analysis_v10_eQTL_updated/Ner

In [14]:
# 6. find breast tissue eQTL file

breast_files  = [m for m in members if 'breast' in m.lower() or 'Breast' in m]
breast_signif = [f for f in breast_files if 'signif' in f.lower() or 'significant' in f.lower()]
if not breast_signif:
    breast_signif = breast_files

print('breast files:', breast_files)
print('using:', breast_signif[0])

breast files: ['GTEx_Analysis_v10_eQTL_updated/Breast_Mammary_Tissue.v10.eGenes.txt.gz', 'GTEx_Analysis_v10_eQTL_updated/Breast_Mammary_Tissue.v10.eQTLs.signif_pairs.parquet']
using: GTEx_Analysis_v10_eQTL_updated/Breast_Mammary_Tissue.v10.eQTLs.signif_pairs.parquet


In [15]:
# 7. load breast eQTL

target_file = breast_signif[0]
print('loading:', target_file)

with tarfile.open(gtex_file, 'r') as tar:
    member = tar.getmember(target_file)
    f      = tar.extractfile(member)
    import io
    content = io.BytesIO(f.read())
    gtex_breast = pd.read_parquet(content)

print('gtex breast eQTL shape:', gtex_breast.shape)
print('columns:', list(gtex_breast.columns))
print(gtex_breast.head(2))

loading: GTEx_Analysis_v10_eQTL_updated/Breast_Mammary_Tissue.v10.eQTLs.signif_pairs.parquet
gtex breast eQTL shape: (2280626, 12)
columns: ['gene_id', 'variant_id', 'tss_distance', 'af', 'ma_samples', 'ma_count', 'pval_nominal', 'slope', 'slope_se', 'pval_nominal_threshold', 'min_pval_nominal', 'pval_beta']
             gene_id           variant_id  tss_distance        af  ma_samples  \
0  ENSG00000227232.5   chr1_64764_C_T_b38         35211  0.070450          70   
1  ENSG00000227232.5  chr1_665098_G_A_b38        635545  0.125245         115   

   ma_count  pval_nominal     slope  slope_se  pval_nominal_threshold  \
0        72  1.837744e-07  0.563176  0.106268                0.000292   
1       128  2.003120e-08  0.453167  0.079270                0.000292   

   min_pval_nominal  pval_beta  
0      2.003120e-08   0.000022  
1      2.003120e-08   0.000022  


In [16]:
# 8. cross-reference SV genes with eQTL

# find gene column
gene_col = next((c for c in gtex_breast.columns if 'gene' in c.lower()), gtex_breast.columns[1])
print('gene column:', gene_col)
print('sample gene ids:', gtex_breast[gene_col].head(3).tolist())

# GTEx uses ENSEMBL IDs, need to map to symbols
# strip version number from ensembl id
gtex_breast['gene_id_clean'] = gtex_breast[gene_col].str.split('.').str[0]

gene column: gene_id
sample gene ids: ['ENSG00000227232.5', 'ENSG00000227232.5', 'ENSG00000227232.5']


In [17]:
# 9. map ensembl to gene symbols

import mygene
mg = mygene.MyGeneInfo()

ensembl_ids = gtex_breast['gene_id_clean'].unique().tolist()
print('unique ensembl ids:', len(ensembl_ids))

# query in batches
result = mg.querymany(
    ensembl_ids[:5000],
    scopes='ensembl.gene',
    fields='symbol',
    species='human',
    as_dataframe=True
)

ensembl_to_symbol = result['symbol'].dropna().to_dict()
print('mapped ids:', len(ensembl_to_symbol))

gtex_breast['gene_symbol'] = gtex_breast['gene_id_clean'].map(ensembl_to_symbol)
panel_in_gtex = [g for g in all_panel if g in gtex_breast['gene_symbol'].values]
print('panel genes with breast eQTL:', len(panel_in_gtex))
print('genes:', panel_in_gtex)

unique ensembl ids: 13415


4 input query terms found dup hits:	[('ENSG00000261600', 2), ('ENSG00000227110', 2), ('ENSG00000215156', 2), ('ENSG00000285761', 3)]
88 input query terms found no hit:	['ENSG00000226849', 'ENSG00000261135', 'ENSG00000288982', 'ENSG00000288573', 'ENSG00000234810', 'ENS


mapped ids: 4304
panel genes with breast eQTL: 3
genes: ['HORMAD1', 'MSH4', 'MSH5']


In [18]:
# 10. eQTL evidence at SV genes

pval_col = next((c for c in gtex_breast.columns if 'pval' in c.lower() or 'p_value' in c.lower()), None)
slope_col = next((c for c in gtex_breast.columns if 'slope' in c.lower() or 'beta' in c.lower() or 'effect' in c.lower()), None)

print('pval col:', pval_col)
print('slope col:', slope_col)

eqtl_summary = []
for gene in all_panel:
    subset = gtex_breast[gtex_breast['gene_symbol'] == gene]
    if len(subset) == 0:
        eqtl_summary.append({'gene': gene, 'n_eqtl': 0, 'min_pval': None, 'has_eqtl': 'no'})
    else:
        min_p = subset[pval_col].min() if pval_col else None
        eqtl_summary.append({
            'gene': gene,
            'n_eqtl': len(subset),
            'min_pval': min_p,
            'has_eqtl': 'yes'
        })

eqtl_df = pd.DataFrame(eqtl_summary).set_index('gene')
print('genes with breast eQTL evidence:')
print(eqtl_df[eqtl_df['has_eqtl']=='yes'][['n_eqtl','min_pval']])
eqtl_df.to_csv(os.path.join(tables_dir, 'gtex_eqtl_panel_genes.csv'))

pval col: pval_nominal
slope col: slope
genes with breast eQTL evidence:
         n_eqtl      min_pval
gene                         
HORMAD1     527  8.558393e-47
MSH4        797  1.146667e-33
MSH5         10  1.147925e-05


In [19]:
# 11. SV loci with eQTL support

sv_eqtl_genes = [g for g in sv_genes if g in panel_in_gtex]
print('SV genes with breast eQTL support:', len(sv_eqtl_genes))
print('genes:', sv_eqtl_genes)
print('fraction of SV genes:', round(len(sv_eqtl_genes)/len(sv_genes), 3))

SV genes with breast eQTL support: 3
genes: ['HORMAD1', 'MSH4', 'MSH5']
fraction of SV genes: 0.13


In [20]:
# 12. inspect GSE176078 tar

sc_file = os.path.join(data_dir, 'GSE176078_Wu_etal_2021_BRCA_scRNASeq.tar.gz')

with tarfile.open(sc_file, 'r:gz') as tar:
    sc_members = tar.getnames()

print('files in scRNA tar:', len(sc_members))
for m in sc_members:
    print(' ', m)

files in scRNA tar: 5
  Wu_etal_2021_BRCA_scRNASeq
  Wu_etal_2021_BRCA_scRNASeq/count_matrix_sparse.mtx
  Wu_etal_2021_BRCA_scRNASeq/count_matrix_genes.tsv
  Wu_etal_2021_BRCA_scRNASeq/count_matrix_barcodes.tsv
  Wu_etal_2021_BRCA_scRNASeq/metadata.csv


In [26]:
# 13. set sc paths

sc_extract_dir = os.path.join(base, 'results', 'sc_extracted')
sc_dir = os.path.join(sc_extract_dir, 'Wu_etal_2021_BRCA_scRNASeq')
mtx_path = os.path.join(sc_dir, 'count_matrix_sparse.mtx')

print('sc_dir:', sc_dir)
print('mtx exists:', os.path.exists(mtx_path))

sc_dir: D:/TNBC_SV_DNA_Repair\results\sc_extracted\Wu_etal_2021_BRCA_scRNASeq
mtx exists: True


In [27]:
# 14. load scRNA barcodes and genes

sc_dir = os.path.join(base, 'results', 'sc_extracted', 'Wu_etal_2021_BRCA_scRNASeq')

barcodes = pd.read_csv(os.path.join(sc_dir, 'count_matrix_barcodes.tsv'), header=None, sep='\t')
genes    = pd.read_csv(os.path.join(sc_dir, 'count_matrix_genes.tsv'),    header=None, sep='\t')
metadata = pd.read_csv(os.path.join(sc_dir, 'metadata.csv'), index_col=0)

print('barcodes:', barcodes.shape)
print('genes:', genes.shape)
print('metadata:', metadata.shape)
print('metadata columns:', list(metadata.columns))
print(metadata.head(3))

barcodes: (100064, 1)
genes: (29733, 1)
metadata: (100064, 8)
metadata columns: ['orig.ident', 'nCount_RNA', 'nFeature_RNA', 'percent.mito', 'subtype', 'celltype_subset', 'celltype_minor', 'celltype_major']
                         orig.ident  nCount_RNA  nFeature_RNA  percent.mito  \
CID3586_AAGACCTCAGCATGAG    CID3586        4581          1689      1.506221   
CID3586_AAGGTTCGTAGTACCT    CID3586        1726           779      5.793743   
CID3586_ACCAGTAGTTGTGGCC    CID3586        1229           514      1.383238   

                         subtype    celltype_subset     celltype_minor  \
CID3586_AAGACCTCAGCATGAG   HER2+  Endothelial ACKR1  Endothelial ACKR1   
CID3586_AAGGTTCGTAGTACCT   HER2+  Endothelial ACKR1  Endothelial ACKR1   
CID3586_ACCAGTAGTTGTGGCC   HER2+  Endothelial ACKR1  Endothelial ACKR1   

                         celltype_major  
CID3586_AAGACCTCAGCATGAG    Endothelial  
CID3586_AAGGTTCGTAGTACCT    Endothelial  
CID3586_ACCAGTAGTTGTGGCC    Endothelial  


In [28]:
# 15. check metadata columns

celltype_col = next((c for c in metadata.columns if any(
    x in c.lower() for x in ['celltype','cell_type','cluster','annotation']
)), None)

patient_col = next((c for c in metadata.columns if any(
    x in c.lower() for x in ['patient','sample','donor','subject','orig']
)), None)

print('cell type column:', celltype_col)
print('patient column:', patient_col)

if celltype_col:
    print('cell types:', metadata[celltype_col].value_counts().head(10))
if patient_col:
    print('patients:', metadata[patient_col].nunique())

cell type column: celltype_subset
patient column: orig.ident
cell types: celltype_subset
T_cells_c1_CD4+_IL7R            7786
Cancer LumA SC                  7742
Cancer Cycling                  5359
T_cells_c4_CD8+_ZFP36           5063
T_cells_c0_CD4+_CCR7            4952
Endothelial ACKR1               4611
Cancer Basal SC                 4312
T_cells_c2_CD4+_T-regs_FOXP3    4246
Cancer Her2 SC                  3708
Plasmablasts                    3524
Name: count, dtype: int64
patients: 26


In [29]:
# 16. read MTX header only

with open(mtx_path, 'r') as f:
    # skip comments
    while True:
        line = f.readline()
        if not line.startswith('%'):
            break
    n_rows, n_cols, n_entries = map(int, line.strip().split())

print('matrix dimensions:', n_rows, 'x', n_cols)
print('non-zero entries:', n_entries)
print('genes count:', len(genes))
print('barcodes count:', len(barcodes))

# figure out orientation
if n_rows == len(genes):
    print('orientation: genes x cells')
    gene_axis = 0
elif n_rows == len(barcodes):
    print('orientation: cells x genes')
    gene_axis = 1

matrix dimensions: 29733 x 100064
non-zero entries: 177994136
genes count: 29733
barcodes count: 100064
orientation: genes x cells


In [30]:
# 17. find immune gene row indices

gene_names  = genes[0].tolist()
cell_names  = barcodes[0].tolist()

immune_idx_set = {
    i: gene_names[i]
    for i in range(len(gene_names))
    if gene_names[i] in all_immune_genes
}

print('immune gene indices found:', len(immune_idx_set))
print('genes:', list(immune_idx_set.values()))

immune gene indices found: 31
genes: ['CD2', 'IL10', 'GNLY', 'CD8A', 'CD8B', 'CTLA4', 'PDCD1', 'TIGIT', 'GZMA', 'HAVCR2', 'VEGFA', 'PRDM1', 'ARG1', 'IL6', 'FOXP3', 'IDO1', 'TOX', 'CD274', 'PDCD1LG2', 'PRF1', 'ENTPD1', 'LAG3', 'KLRG1', 'NR4A1', 'IFNG', 'GZMH', 'GZMB', 'BATF', 'STAT3', 'TGFB1', 'NKG7']


In [31]:
# 18. stream MTX and save immune data
# fix: immune_data exists: False
from collections import defaultdict

immune_data = defaultdict(lambda: defaultdict(float))

print('streaming MTX file for immune genes only')

with open(mtx_path, 'r') as f:
    for line in f:
        if line.startswith('%'):
            continue
        parts = line.strip().split()
        if len(parts) == 3 and parts[0].isdigit():
            n_rows_h, n_cols_h, _ = int(parts[0]), int(parts[1]), int(parts[2])
            break

    line_count = 0
    for line in f:
        parts = line.strip().split()
        if len(parts) < 3:
            continue
        r, c, v = int(parts[0])-1, int(parts[1])-1, float(parts[2])

        if gene_axis == 0:
            gene_i, cell_i = r, c
        else:
            gene_i, cell_i = c, r

        if gene_i in immune_idx_set:
            gene_name = immune_idx_set[gene_i]
            immune_data[gene_name][cell_i] += v

        line_count += 1
        if line_count % 5000000 == 0:
            print(f'  processed {line_count//1000000}M entries')

print('done streaming')

# immediately build and save to disk
n_cells = len(cell_names)
immune_expr_sc = pd.DataFrame(index=cell_names)
for gene, cell_dict in immune_data.items():
    col = np.zeros(n_cells)
    for cell_i, val in cell_dict.items():
        if cell_i < n_cells:
            col[cell_i] = val
    immune_expr_sc[gene] = col

save_path = os.path.join(scores_dir, 'immune_expr_sc.csv')
immune_expr_sc.to_csv(save_path)
print('saved to disk:', save_path)
print('shape:', immune_expr_sc.shape)

streaming MTX file for immune genes only
  processed 5M entries
  processed 10M entries
  processed 15M entries
  processed 20M entries
  processed 25M entries
  processed 30M entries
  processed 35M entries
  processed 40M entries
  processed 45M entries
  processed 50M entries
  processed 55M entries
  processed 60M entries
  processed 65M entries
  processed 70M entries
  processed 75M entries
  processed 80M entries
  processed 85M entries
  processed 90M entries
  processed 95M entries
  processed 100M entries
  processed 105M entries
  processed 110M entries
  processed 115M entries
  processed 120M entries
  processed 125M entries
  processed 130M entries
  processed 135M entries
  processed 140M entries
  processed 145M entries
  processed 150M entries
  processed 155M entries
  processed 160M entries
  processed 165M entries
  processed 170M entries
  processed 175M entries
done streaming
saved to disk: D:/TNBC_SV_DNA_Repair\results\scores\immune_expr_sc.csv
shape: (100064, 31

In [32]:
# 19. normalize immune expression

# log1p normalize
immune_expr_sc_norm = np.log1p(immune_expr_sc)

print('immune expression shape:', immune_expr_sc_norm.shape)
print('value range:', round(immune_expr_sc_norm.values.min(),3), 'to', round(immune_expr_sc_norm.values.max(),3))

immune expression shape: (100064, 31)
value range: 0.0 to 5.864


In [33]:
# 20. filter T cells and score

meta_aligned = metadata.loc[metadata.index.isin(immune_expr_sc_norm.index)]

if celltype_col:
    t_mask = meta_aligned[celltype_col].astype(str).str.contains(
        'T.cell|Tcell|CD8|CD4|Exhaust|NK|Cytotox', case=False, na=False
    )
    t_cells = meta_aligned[t_mask].index.tolist()
else:
    t_cells = meta_aligned.index.tolist()

mat_t = immune_expr_sc_norm.loc[t_cells]
print('T/NK cells:', mat_t.shape)

exhaust_use  = [g for g in exhaustion_genes     if g in mat_t.columns]
cytotox_use  = [g for g in cytotoxic_genes      if g in mat_t.columns]
suppress_use = [g for g in immune_suppress_genes if g in mat_t.columns]

mat_t = mat_t.copy()
mat_t['exhaustion_score'] = mat_t[exhaust_use].mean(axis=1)
mat_t['cytotoxic_score']  = mat_t[cytotox_use].mean(axis=1)
mat_t['suppress_score']   = mat_t[suppress_use].mean(axis=1)

if 'PDCD1' in mat_t.columns and 'CD2' in mat_t.columns:
    mat_t['pdcd1_cd2_ratio'] = mat_t['PDCD1'] / (mat_t['CD2'] + 1e-6)

print('scores computed')
print('exhaustion mean:', round(mat_t['exhaustion_score'].mean(), 4))
print('cytotoxic mean:', round(mat_t['cytotoxic_score'].mean(), 4))

T/NK cells: (35214, 31)
scores computed
exhaustion mean: 0.1804
cytotoxic mean: 0.3125


In [34]:
# 21. aggregate per patient

mat_t[patient_col] = meta_aligned.loc[t_cells, patient_col]

agg_cols = ['exhaustion_score','cytotoxic_score','suppress_score','PDCD1','CD2','pdcd1_cd2_ratio']
agg_cols = [c for c in agg_cols if c in mat_t.columns]

sc_patient = mat_t.groupby(patient_col)[agg_cols].mean()

print('patient signatures:', sc_patient.shape)
print(sc_patient.head(3))
sc_patient.to_csv(os.path.join(tables_dir, 'sc_patient_immune_scores.csv'))

patient signatures: (26, 6)
            exhaustion_score  cytotoxic_score  suppress_score     PDCD1  \
orig.ident                                                                
CID3586             0.115212         0.171545        0.061764  0.057678   
CID3838             0.393386         0.328326        0.215301  0.197710   
CID3921             0.247059         0.246545        0.090465  0.135182   

                 CD2  pdcd1_cd2_ratio  
orig.ident                             
CID3586     0.460614     23129.228417  
CID3838     1.548020      5943.923046  
CID3921     0.731580     28936.978351  


In [35]:
# 22. signature summary

print('exhaustion genes used:', exhaust_use)
print('cytotoxic genes used:', cytotox_use)
print('suppression genes used:', suppress_use)
print('PDCD1 found:', 'PDCD1' in mat_t.columns)
print('CD2 found:', 'CD2' in mat_t.columns)
print('patients in sc data:', sc_patient.shape[0])

exhaustion genes used: ['PDCD1', 'LAG3', 'HAVCR2', 'TIGIT', 'CTLA4', 'TOX', 'NR4A1', 'ENTPD1', 'BATF', 'PRDM1']
cytotoxic genes used: ['GZMB', 'PRF1', 'IFNG', 'CD8A', 'CD8B', 'GNLY', 'NKG7', 'GZMA', 'GZMH', 'KLRG1']
suppression genes used: ['FOXP3', 'IL10', 'TGFB1', 'CD274', 'PDCD1LG2', 'IDO1', 'ARG1', 'VEGFA', 'IL6', 'STAT3']
PDCD1 found: True
CD2 found: True
patients in sc data: 26


In [36]:
# 23. build gene sets for ssGSEA

gene_sets_ssgsea = {}

exhaust_valid  = [g for g in exhaustion_genes     if g in immune_expr_df.index] if 'immune_expr_df' in dir() else []
cytotox_valid  = [g for g in cytotoxic_genes      if g in immune_expr_df.index] if 'immune_expr_df' in dir() else []
suppress_valid = [g for g in immune_suppress_genes if g in immune_expr_df.index] if 'immune_expr_df' in dir() else []

# use sc-derived gene sets based on what was found
exhaust_valid  = exhaust_use
cytotox_valid  = cytotox_use
suppress_valid = suppress_use

if len(exhaust_valid)  >= 2: gene_sets_ssgsea['exhaustion']  = exhaust_valid
if len(cytotox_valid)  >= 2: gene_sets_ssgsea['cytotoxic']   = cytotox_valid
if len(suppress_valid) >= 2: gene_sets_ssgsea['suppression'] = suppress_valid
if 'PDCD1' in mat_t.columns: gene_sets_ssgsea['PDCD1_single'] = ['PDCD1']
if 'CD2'   in mat_t.columns: gene_sets_ssgsea['CD2_single']   = ['CD2']

for k, v in gene_sets_ssgsea.items():
    print(f'{k}: {v}')

exhaustion: ['PDCD1', 'LAG3', 'HAVCR2', 'TIGIT', 'CTLA4', 'TOX', 'NR4A1', 'ENTPD1', 'BATF', 'PRDM1']
cytotoxic: ['GZMB', 'PRF1', 'IFNG', 'CD8A', 'CD8B', 'GNLY', 'NKG7', 'GZMA', 'GZMH', 'KLRG1']
suppression: ['FOXP3', 'IL10', 'TGFB1', 'CD274', 'PDCD1LG2', 'IDO1', 'ARG1', 'VEGFA', 'IL6', 'STAT3']
PDCD1_single: ['PDCD1']
CD2_single: ['CD2']


In [37]:
# 24. load TCGA immune expression

tcga_expr_file = os.path.join(data_dir, 'TCGA.BRCA.sampleMap_HiSeqV2_exon.gz')

import mygene as mg_module
mg2 = mg_module.MyGeneInfo()

immune_query = mg2.querymany(
    all_immune_genes,
    scopes='symbol',
    fields='symbol,genomic_pos',
    species='human',
    as_dataframe=True
)
immune_query = immune_query[~immune_query.index.duplicated(keep='first')]
print('immune genes queried:', len(immune_query))
print('with coords:', immune_query['genomic_pos.chr'].notna().sum())

immune genes queried: 31
with coords: 30


In [38]:
# 25. extract immune gene expression from TCGA

immune_expr_rows = {}

with gzip.open(tcga_expr_file, 'rt') as f:
    header_line = f.readline().rstrip().split('\t')
    sample_list = header_line[1:]
    tnbc_idx    = [i+1 for i, s in enumerate(sample_list) if s in tnbc_samples]
    tnbc_names  = [sample_list[i-1] for i in tnbc_idx]

    for line in f:
        parts = line.rstrip().split('\t')
        coord = parts[0]
        try:
            c  = coord.split(':')[0].replace('chr','')
            se = coord.split(':')[1].split('-')
            s  = int(se[0])
            e  = int(se[1])
        except:
            continue
        for gene in all_immune_genes:
            if gene not in immune_query.index:
                continue
            row = immune_query.loc[gene]
            gc  = str(row.get('genomic_pos.chr',''))
            gs  = row.get('genomic_pos.start', None)
            ge  = row.get('genomic_pos.end',   None)
            if pd.isna(gs) or pd.isna(ge):
                continue
            if c == gc and not (e < int(gs) or s > int(ge)):
                vals = [float(parts[i]) if i < len(parts) else 0.0 for i in tnbc_idx]
                if gene not in immune_expr_rows:
                    immune_expr_rows[gene] = vals
                else:
                    immune_expr_rows[gene] = [max(a,b) for a,b in zip(immune_expr_rows[gene], vals)]

immune_expr_df = pd.DataFrame(immune_expr_rows, index=tnbc_names).T
immune_expr_df = np.log2(immune_expr_df + 1)

print('TCGA immune expression:', immune_expr_df.shape)
print('genes found:', list(immune_expr_df.index))

TCGA immune expression: (18, 121)
genes found: ['STAT3', 'CD8A', 'GZMB', 'ARG1', 'ENTPD1', 'CD8B', 'CD274', 'BATF', 'KLRG1', 'TGFB1', 'GZMH', 'IL10', 'PDCD1LG2', 'HAVCR2', 'TOX', 'GZMA', 'LAG3', 'NR4A1']


In [42]:
# 26. run ssGSEA

import gseapy as gp

gs_valid = {
    k: [g for g in v if g in immune_expr_df.index]
    for k, v in gene_sets_ssgsea.items()
}
gs_valid = {k: v for k, v in gs_valid.items() if len(v) >= 1}

print('gene sets for ssGSEA:')
for k, v in gs_valid.items():
    print(f'  {k}: {v}')

ssgsea_result = gp.ssgsea(
    data=immune_expr_df,
    gene_sets=gs_valid,
    outdir=None,
    sample_norm_method='rank',
    no_plot=True,
    min_size=1
)

ssgsea_scores = ssgsea_result.res2d.pivot(
    index='Term', columns='Name', values='NES'
).T

print('ssGSEA scores shape:', ssgsea_scores.shape)
print('columns:', list(ssgsea_scores.columns))

gene sets for ssGSEA:
  exhaustion: ['LAG3', 'HAVCR2', 'TOX', 'NR4A1', 'ENTPD1', 'BATF']
  cytotoxic: ['GZMB', 'CD8A', 'CD8B', 'GZMA', 'GZMH', 'KLRG1']
  suppression: ['IL10', 'TGFB1', 'CD274', 'PDCD1LG2', 'ARG1', 'STAT3']
ssGSEA scores shape: (121, 3)
columns: ['cytotoxic', 'exhaustion', 'suppression']


In [43]:
# 27. PDCD1 CD2 ratio bulk

if 'PDCD1' in immune_expr_df.index and 'CD2' in immune_expr_df.index:
    pdcd1 = immune_expr_df.loc['PDCD1']
    cd2   = immune_expr_df.loc['CD2']
    ratio = pdcd1 / (cd2 + 1e-6)
    bulk_immune = ssgsea_scores.copy()
    bulk_immune['PDCD1']           = pdcd1.values
    bulk_immune['CD2']             = cd2.values
    bulk_immune['pdcd1_cd2_ratio'] = ratio.values
else:
    bulk_immune = ssgsea_scores.copy()

bulk_immune.index.name = 'sample'
bulk_immune = bulk_immune.reset_index()

print('bulk immune scores:', bulk_immune.shape)
print('columns:', list(bulk_immune.columns))

bulk immune scores: (121, 4)
columns: ['sample', 'cytotoxic', 'exhaustion', 'suppression']


In [44]:
# 28. merge with GIPS

immune_gips = gips_df.merge(bulk_immune, on='sample', how='inner')
print('merged shape:', immune_gips.shape)
print('GIPS groups:', immune_gips['GIPS_group'].value_counts().to_dict())
immune_gips.to_csv(os.path.join(tables_dir, 'immune_gips_merged.csv'), index=False)

merged shape: (121, 11)
GIPS groups: {'Low': 52, 'High': 36, 'Moderate': 33}


In [47]:
# 29. statistical tests

from scipy.stats import kruskal, mannwhitneyu

# force numeric
for col in ['cytotoxic', 'exhaustion', 'suppression']:
    immune_gips[col] = pd.to_numeric(immune_gips[col], errors='coerce')

immune_cols = ['cytotoxic', 'exhaustion', 'suppression']
if 'pdcd1_cd2_ratio' in immune_gips.columns:
    immune_gips['pdcd1_cd2_ratio'] = pd.to_numeric(immune_gips['pdcd1_cd2_ratio'], errors='coerce')
    immune_cols.append('pdcd1_cd2_ratio')
if 'PDCD1' in immune_gips.columns:
    immune_gips['PDCD1'] = pd.to_numeric(immune_gips['PDCD1'], errors='coerce')
    immune_cols.append('PDCD1')
if 'CD2' in immune_gips.columns:
    immune_gips['CD2'] = pd.to_numeric(immune_gips['CD2'], errors='coerce')
    immune_cols.append('CD2')

group_order = ['Low', 'Moderate', 'High']

stat_results = []
for col in immune_cols:
    try:
        low  = immune_gips[immune_gips['GIPS_group']=='Low'][col].dropna().astype(float).values
        mod  = immune_gips[immune_gips['GIPS_group']=='Moderate'][col].dropna().astype(float).values
        high = immune_gips[immune_gips['GIPS_group']=='High'][col].dropna().astype(float).values
        h, p = kruskal(low, mod, high)
        lo_hi = mannwhitneyu(low, high, alternative='two-sided')
        stat_results.append({
            'metric':            col,
            'kruskal_H':         round(h, 3),
            'kruskal_p':         round(p, 4),
            'mwu_low_vs_high_p': round(lo_hi.pvalue, 4)
        })
        print(f'{col}: H={round(h,3)} p={round(p,4)}')
    except Exception as ex:
        print(f'{col}: error {ex}')

stat_df = pd.DataFrame(stat_results).sort_values('kruskal_p')
print(stat_df)
stat_df.to_csv(os.path.join(tables_dir, 'immune_gips_stats.csv'), index=False)

cytotoxic: H=0.516 p=0.7725
exhaustion: H=5.339 p=0.0693
suppression: H=1.634 p=0.4417
        metric  kruskal_H  kruskal_p  mwu_low_vs_high_p
1   exhaustion      5.339     0.0693             0.1087
2  suppression      1.634     0.4417             0.5696
0    cytotoxic      0.516     0.7725             0.5244


In [48]:
# 30. violin plots

group_colors = {'Low': '#4878cf', 'Moderate': '#f0a500', 'High': '#d94f3d'}
plot_cols    = [c for c in immune_cols if c in immune_gips.columns][:6]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    ax        = axes[i]
    data_plot = [
        immune_gips[immune_gips['GIPS_group']==g][col].dropna().values
        for g in group_order
    ]
    parts = ax.violinplot(data_plot, positions=[1,2,3], showmedians=True)
    for pc, g in zip(parts['bodies'], group_order):
        pc.set_facecolor(group_colors[g])
        pc.set_alpha(0.7)
    ax.set_xticks([1,2,3])
    ax.set_xticklabels(group_order, fontsize=9)
    ax.set_title(col, fontsize=10)
    ax.set_ylabel('score', fontsize=8)
    row = stat_df[stat_df['metric']==col]
    if len(row) > 0:
        ax.set_xlabel(f'p={row["kruskal_p"].values[0]}', fontsize=8)

for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Immune scores by GIPS group', fontsize=12)
plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb3_immune_by_gips.png'), dpi=150)
plt.show()
print('saved violin plots')

saved violin plots


In [49]:
# 31. PDCD1 CD2 plot

if 'pdcd1_cd2_ratio' in immune_gips.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    data_box = [
        immune_gips[immune_gips['GIPS_group']==g]['pdcd1_cd2_ratio'].dropna().values
        for g in group_order
    ]
    bp = axes[0].boxplot(data_box, patch_artist=True, widths=0.5)
    for patch, g in zip(bp['boxes'], group_order):
        patch.set_facecolor(group_colors[g])
        patch.set_alpha(0.7)
    axes[0].set_xticklabels(group_order)
    axes[0].set_ylabel('PDCD1/CD2 ratio')
    axes[0].set_title('PDCD1/CD2 by GIPS group')

    colors = immune_gips['GIPS_group'].map(group_colors)
    axes[1].scatter(
        immune_gips['GIPS_scaled'],
        immune_gips['pdcd1_cd2_ratio'],
        c=colors, alpha=0.6, s=30, edgecolors='none'
    )
    r, p = stats.pearsonr(
        immune_gips['GIPS_scaled'],
        immune_gips['pdcd1_cd2_ratio'].fillna(0)
    )
    axes[1].set_xlabel('GIPS score')
    axes[1].set_ylabel('PDCD1/CD2 ratio')
    axes[1].set_title(f'GIPS vs PDCD1/CD2 (r={round(r,3)}, p={round(p,4)})')

    plt.tight_layout()
    fig.savefig(os.path.join(figures_dir, 'nb3_pdcd1_cd2_gips.png'), dpi=150)
    plt.show()
    print('saved PDCD1/CD2 plot')

In [50]:
# 32. eQTL plot

eqtl_plot        = eqtl_df.copy()
eqtl_plot['has_sv']  = [sv_ev.loc[g,'either_sv'] if g in sv_ev.index else 'no' for g in eqtl_plot.index]
eqtl_plot['n_eqtl']  = pd.to_numeric(eqtl_plot['n_eqtl'], errors='coerce').fillna(0)
eqtl_plot['group']   = ['HR' if g in hr_genes else 'cohesin' if g in cohesin_genes else 'meiosis' for g in eqtl_plot.index]
eqtl_plot_sorted     = eqtl_plot.sort_values(['group','n_eqtl'], ascending=[True,False])

bar_colors = [
    '#d94f3d' if row['has_sv']=='yes' else '#aaaaaa'
    for _, row in eqtl_plot_sorted.iterrows()
]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(eqtl_plot_sorted))
ax.bar(x, eqtl_plot_sorted['n_eqtl'], color=bar_colors, edgecolor='none')
ax.set_xticks(x)
ax.set_xticklabels(eqtl_plot_sorted.index, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('breast tissue eQTL count')
ax.set_title('GTEx v10 eQTL evidence at panel genes (red = SV candidate)')

plt.tight_layout()
fig.savefig(os.path.join(figures_dir, 'nb3_eqtl_evidence.png'), dpi=150)
plt.show()
print('saved eQTL plot')

saved eQTL plot


In [51]:
# 33. save all results

immune_gips.to_csv(os.path.join(tables_dir, 'immune_gips_merged.csv'), index=False)
stat_df.to_csv(os.path.join(tables_dir, 'immune_gips_stats.csv'), index=False)
eqtl_df.to_csv(os.path.join(tables_dir, 'gtex_eqtl_panel_genes.csv'))
print('all results saved')

all results saved


In [52]:
# 34. notebook 3 summary

summary3 = {
    'GTEx breast eQTL panel genes':  len(panel_in_gtex),
    'SV genes with eQTL support':    len(sv_eqtl_genes),
    'scRNA cells processed':         mat_t.shape[0],
    'ssGSEA gene sets':              len(gs_valid),
    'immune GIPS merged samples':    len(immune_gips),
    'significant immune metrics':    int((stat_df['kruskal_p'] < 0.05).sum()),
}

for k, v in summary3.items():
    print(f'{k}: {v}')

pd.DataFrame.from_dict(
    summary3, orient='index', columns=['value']
).to_csv(os.path.join(tables_dir, 'nb3_summary.csv'))

print('notebook 3 complete')

GTEx breast eQTL panel genes: 3
SV genes with eQTL support: 3
scRNA cells processed: 35214
ssGSEA gene sets: 3
immune GIPS merged samples: 121
significant immune metrics: 0
notebook 3 complete
